# NBA Golden Dataset — Upload

This notebook creates the offline evaluation dataset for the multi-turn NBA credit-card chatbot.

Each example is a **scripted conversation** — a list of user turns the evaluator will replay against the graph one turn at a time. The reference output records which offer we expect the bot to end up recommending (or that we expect the fraud guardrail to fire).

Runs against LangSmith via `client.create_dataset` + `client.create_examples`, mirroring the pattern in `dataset_upload.ipynb` from the LangSmith course.

## Setup

In [ ]:
import os
# You can set these inline or via a .env file.
# os.environ['LANGSMITH_API_KEY'] = '...'
os.environ.setdefault('LANGSMITH_TRACING', 'true')
os.environ.setdefault('LANGSMITH_PROJECT', 'nba-demo')

from dotenv import load_dotenv
load_dotenv(override=True)

In [ ]:
from langsmith import Client

client = Client()
DATASET_NAME = 'NBA Golden Dataset'

## Define scripted conversations

Each example is:

```python
inputs  = {'turns': [<user turn 1>, <user turn 2>, ...], 'customer_id': <int>}
outputs = {'expected_offer_id': 'PLAT_TRAVEL', 'expected_path': 'proceed', 'min_turns_to_offer': 2, 'max_turns_to_offer': 4}
```

The `split` label lets us slice experiments by offer type from `nba_experiments.ipynb`.

In [ ]:
EXAMPLES = [
    # ---- travel ----
    dict(split='travel', customer_id=101,
         turns=[
            "Hi, I'm thinking about upgrading my credit card.",
            "I travel a lot for work — I fly maybe twice a month. I earn about $180,000 a year and I'm employed full-time.",
         ], expected_offer_id='PLAT_TRAVEL', expected_path='proceed'),
    dict(split='travel', customer_id=102,
         turns=[
            "Looking for a premium card.",
            "I'd love lounge access and points on hotels. Annual income is around $150K.",
         ], expected_offer_id='PLAT_TRAVEL', expected_path='proceed'),
    dict(split='travel', customer_id=103,
         turns=[
            "I want to earn miles.",
            "I'm 34, employed, make $120K, and I spend a lot on flights and restaurants.",
         ], expected_offer_id='PLAT_TRAVEL', expected_path='proceed'),

    # ---- balance-transfer ----
    dict(split='balance-transfer', customer_id=201,
         turns=[
            "I'm carrying some credit card debt and want to consolidate.",
            "I owe about $8,000 across two cards. I make $55K a year, employed.",
         ], expected_offer_id='BALANCE_TRANSFER', expected_path='proceed'),
    dict(split='balance-transfer', customer_id=202,
         turns=[
            "Any card that can help me pay less interest?",
            "I've got roughly $12,000 in existing debt, income around $60K.",
         ], expected_offer_id='BALANCE_TRANSFER', expected_path='proceed'),
    dict(split='balance-transfer', customer_id=203,
         turns=[
            "Need to consolidate a couple of balances.",
            "I'm 45, employed, earning about $70K, and I have $10K existing debt.",
         ], expected_offer_id='BALANCE_TRANSFER', expected_path='proceed'),

    # ---- secured builder ----
    dict(split='secured', customer_id=301,
         turns=[
            "I want to rebuild my credit.",
            "I had some late payments last year. I'm employed and make around $30K.",
         ], expected_offer_id='SECURED_BUILDER', expected_path='proceed'),
    dict(split='secured', customer_id=302,
         turns=[
            "My credit score is pretty low, I need help.",
            "I'd put down a deposit if it helps. Income is about $25K.",
         ], expected_offer_id='SECURED_BUILDER', expected_path='proceed'),

    # ---- student ----
    dict(split='student', customer_id=401,
         turns=[
            "I'm a college student and I want to start building credit.",
            "I'm 20, in school, and I work part-time earning maybe $8K a year.",
         ], expected_offer_id='STUDENT_STARTER', expected_path='proceed'),
    dict(split='student', customer_id=402,
         turns=[
            "Looking for my first credit card as a student.",
            "I'm 22, studying full-time, no income to speak of.",
         ], expected_offer_id='STUDENT_STARTER', expected_path='proceed'),

    # ---- cashback / generic ----
    dict(split='cashback', customer_id=501,
         turns=[
            "I just want a simple card with rewards.",
            "I earn about $65K and spend on everyday stuff — groceries, gas.",
         ], expected_offer_id='CASHBACK_EVERYDAY', expected_path='proceed'),
    dict(split='cashback', customer_id=502,
         turns=[
            "Nothing fancy, just a card with cash back.",
            "I'm 40, employed, income $50K.",
         ], expected_offer_id='CASHBACK_EVERYDAY', expected_path='proceed'),

    # ---- fraud decline ----
    dict(split='fraud-decline', customer_id=601,
         turns=[
            "I want to buy a new car — putting through a $25,000 charge tonight.",
            "I'm at longitude 45 and latitude 85, and my current transaction is for $25,000.",
         ], expected_offer_id=None, expected_path='decline'),
    dict(split='fraud-decline', customer_id=602,
         turns=[
            "Big transaction, need it approved right now.",
            "Transaction amount is $12,000, longitude 200 latitude 200 — pushing it through now.",
         ], expected_offer_id=None, expected_path='decline'),
]

print(f'{len(EXAMPLES)} examples ready to upload.')

## Create the dataset (idempotent)

In [ ]:
existing = list(client.list_datasets(dataset_name=DATASET_NAME))
if existing:
    dataset = existing[0]
    print(f'Reusing existing dataset {dataset.id}')
else:
    dataset = client.create_dataset(
        dataset_name=DATASET_NAME,
        description='Scripted multi-turn customer conversations for the NBA credit-card chatbot demo.',
    )
    print(f'Created dataset {dataset.id}')

In [ ]:
# We upload as individual examples so we can attach a `split` per example.
for ex in EXAMPLES:
    inputs = {'turns': ex['turns'], 'customer_id': ex['customer_id']}
    outputs = {
        'expected_offer_id': ex['expected_offer_id'],
        'expected_path': ex['expected_path'],
        'min_turns_to_offer': 2,
        'max_turns_to_offer': 4,
    }
    client.create_example(
        inputs=inputs,
        outputs=outputs,
        dataset_id=dataset.id,
        split=ex['split'],
    )
print(f'Uploaded {len(EXAMPLES)} examples.')

## Verify

In [ ]:
examples = list(client.list_examples(dataset_name=DATASET_NAME))
print(f'Total examples in `{DATASET_NAME}`: {len(examples)}')
for e in examples[:3]:
    print(e.inputs.get('turns'), '->', e.outputs.get('expected_offer_id'))